# Visualisierung und Vorverarbeitung

In diesem Notebook behandeln wir Best Practices zur Datenvisualisierung sowie einen Aspekt der Datenvorverarbeitung – die Imputation fehlender Werte. Der Datensatz, auf dem wir uns konzentrieren, ist **UCI Heart Disease Dataset (Cleveland-Subset)**. Er umfasst 303 Patienten, 13 klinische Merkmale und eine binäre Zielvariable (Herzerkrankung vorhanden / nicht vorhanden).

## Benötigte Module

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.preprocessing import StandardScaler, MinMaxScaler

## Datensatz zu Herzerkrankungen laden

Der Datensatz zu Herzerkrankungen ([Cleveland Heart Disease](https://archive.ics.uci.edu/dataset/45/heart+disease)) aus dem UCI ML Repository enthält medizinische Messungen von 303 Patienten. Jede Zeile repräsentiert einen Patienten, der durch 13 klinische Merkmale beschrieben wird. Die Zielvariable gibt das Vorhandensein (`1`) oder Fehlen (`0`) einer Herzerkrankung an. Die folgende Tabelle beschreibt alle Spalten der Merkmalmatrix:

| Originalname | Merkmal | Beschreibung    | Datentyp |
|--------------|---------|-----------------|----------|
| `age`      | Alter     | Alter in Jahren | numerisch |
| `sex`      | Geschlecht | 0 = weilblich, 1 = männlich | binär |
| `cp`       | Brustschmerzen | Brustschmerzentyp | kategorial |
| `trestbps` | Ruheblutdruck | Ruheblutdruck (mm Hg) | numerisch |
| `chol`     | Serumcholesterin | Serumcholesterin (mg/dl) | numerisch |
| `fbs`      | Erhöhter Blutzucker | Nüchternblutzucker > 120 mg/dl | binär |
| `restecg`  | Ruhe-EKG | Ruhe-EKGs | kategorial |
| `thalach`  | Maximale Herzfrequenz | Maximale Herzfrequenz erreicht | numerisch |
| `exang`    | Belastungsangina | Belastungsinduzierte Angina | binär |
| `oldpeak`  | ST-Depression | Belastungsinduzierte ST-Senkung | numerisch |
| `slope`    | ST-Segment | Steigung des ST-Segments bei Belastung | kategorial |
| `ca`       | Gefärbte Hauptgefäße | Große Gefäße, mittels Fluoroskopie eingefärbt (0–3) | numerisch |
| `thal`     | Thallium-Herzscan | Nachweis von Ischämie oder Narbengewebe | kategorial |

<span style="font-size:x-small">Weitere Informationen zum Datensatz sind in der folgenden Publikation verfügbar:<br />
R. Detrano et al. [International application of a new probability algorithm for the diagnosis of coronary artery disease](https://www.ajconline.org/article/0002-9149(89)90524-9/pdf). American Journal of Cardiology (1989)</span>

In [ ]:
def parse_dataset(lines: list[str]) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Parsiert den Datensatz zu Herzerkrankungen aus Text.

    :param lines: Inhalt der Textdatei als `list` von `str`-Instanzen, eine pro Zeile.
    :return: Tupel der Form `(x, y, features)`, wobei:
             - `x` Merkmalmatrix als Array der Form `(Beobachtungen, Merkmale)` ist;
             - `y` Ausgabevektor als binärer Array der Form `(Beobachtungen)` ist;
             - `features` Merkmaldefinitionen als `dict`-Object ist, in dem jeder
                  Schlüssel ein Merkmalname ist, und sein Wert - `list` mit Kategorienamen oder
                  einelementige Liste mit der Messeinheit ist.
    """
    delimiter = ','
    columns = {
        'Alter': ['Jahre'],
        'Geschlecht': ['weiblich', 'mänlich'],
        'Brustschmerzen': ['typische Angina', 'atypische Angina', 'nicht-anginös', 'asymptomatisch'],
        'Ruheblutdruck': ['mm Hg'],
        'Serumcholesterin': ['mg/dl'],
        'Erhöhter Blutzucker': ['nein', 'ja'],
        'Ruhe-EKG': ['normal', 'ST-T', 'LVH'],
        'Maximale Herzfrequenz': ['S/min'],
        'Belastungsangina': ['nein', 'ja'],
        'ST-Depression': ['mV'],
        'ST-Segment': ['ansteigend', 'flach', 'abfallend'],
        'Gefärbte Hauptgefäße': ['0', '1', '2', '3'],
        'Thallium-Herzscan': ['normal', 'behobener Defekt', 'reversibler Defekt']
    }
    category_counts = [len(v) for v in columns.values()]
    observation_count, feature_count = len(lines), len(columns)
    x = np.full((observation_count, feature_count), np.nan, dtype=np.float64)
    y = np.empty(observation_count, dtype=np.uint16)
    for i, line in enumerate(lines):
        values = line.strip('\r\n').split(delimiter)
        if len(values) != feature_count + 1:
            raise ValueError(f'Unerwartete Anzahl an Spalten in Zeile {i + 1}')
        for j, value in enumerate(values):
            if j < feature_count:
                if value != '?':
                    if category_counts[j] == 1:
                        x[i, j] = float(value)
                    else:
                        value = int(float(value))
                        if j == 2 or j == 10:
                            value -= 1
                        elif j == 12:
                            value = [3, 6, 7].index(value)
                        if not (0 <= value < category_counts[j]):
                            txt = f'{list(columns.keys())[j]} in Zeile {i + 1}'
                            raise ValueError(f'Unerwartete Kategorieindex für {txt}')
                        x[i, j] = value
            else:
                y[i] = int(value)

    # Zielvariable binarisieren: 0 = keine Erkrankung, 1 = Erkrankung
    y[y != 0] = 1
    return x, y, columns


def extract_lines_from_url(dataset_path: str) -> list[str]:
    """
    Lädt einen Datensatz vom UCI-Repository herunter.

    :param dataset_path: URL zum Datensatz; relativ zum Datenbanken-Ordner.
    :return: Inhalt des Datensatzes als `list` von `str`-Instanzen, eine pro Zeile.
    """
    import urllib
    uci_url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/'
    response = urllib.request.urlopen(uci_url + dataset_path)
    return response.read().decode('utf-8').splitlines()


def extract_lines_from_file(file_path: str) -> list[str]:
    """
    Lädt einen Datensatz aus einer lokalen Datei.

    :return: Inhalt des Datensatzes als `list` von `str`-Instanzen, eine pro Zeile.
    """
    import gzip
    function_open = gzip.open if file_path.endswith('.gz') else open
    with function_open(file_path, 'rt', encoding='utf-8') as file_handle:
        result = file_handle.readlines()
    return result


def print_feature_info(x: np.ndarray, features: dict[str, list]):
    """
    Gibt die Namen und Datentypen aller Features aus.

    :param x: Merkmalmatrix als Array der Form `(Beobachtungen, Merkmale)`.
    :param features: Merkmaldefinitionen als `dict`-Object.
    """
    print("\nMerkmaltypen\n------------")
    for i, (feature_name, feature_info) in enumerate(features.items()):
        category_count = len(feature_info)
        if category_count == 1:
            txt = f'numerisch, [{x[:, i].min()}, {x[:, i].max()}]'
        else:  # type(feature_info) is str:
            txt = f'{category_count} Kategorien: ' + ', '.join(feature_info)
        print(f'{feature_name:>22s} | {txt}')


def print_missing_values(x: np.ndarray, features: dict[str, list]):
    """
    Gibt eine Zusammenfassung der fehlenden Werte pro Merkmal aus.
    
    :param x: Merkmalmatrix als Array der Form `(Beobachtungen, Merkmale)`.
    :param features: Merkmaldefinitionen als `dict`-Object.
    """
    print('\nFehlende Werte\n--------------')
    for i, feature_name in enumerate(features.keys()):
        missing_count = np.isnan(x[:, i]).sum().item()
        print(f'{feature_name:>22s}: {missing_count}')


def print_patient(x: np.ndarray, features: dict[str, list]):
    """
    Gibt alle Merkmalswerte für einen Patienten aus.
    
    :param x: Merkmalvektor für den Patienten / die Patientin.
    :param features: Merkmaldefinitionen als `dict`-Object.
    """
    for (feature_name, categories), value in zip(features.items(), x):
        if len(categories) == 1:
            value = f'{value} {categories[0]}'
        else:  # len(categories) > 1
            value = categories[int(value)]
        print(f'{feature_name:>22s}: {value}')


dataset_url = 'heart-disease/processed.cleveland.data'
x, y, features = parse_dataset(extract_lines_from_url(dataset_url))

print(f'Dimensionen der Markmalsmatrix: {x.shape}')
print(f'Dimensionen des Ausgabevektors: {y.shape}')
print(f'                  Merkmalnamen: {','.join(features.keys())}')
print(f'            Erster Ausgabewert: {y[0]}')
print(f'                   Erste Zeile: {x[0, :]}')

### Lernen wir Patientin X kennen

Nachfolgend wählen wir eine Patientin aus diesem Datensatz aus und verbergen ihre Diagnose.

Schauen Sie sich die klinischen Merkmale an und treffen Sie Ihre Vorhersage: Herzerkrankung oder nicht?

In [ ]:
# Wir wählen die Patientin mit dem Zeilenindex 42 als unsere "Mystery-Patientin" aus
MYSTERY_IDX = 42

print('Klinisches Profil von Pazientin X')
print('---------------------------------')
print_patient(x[MYSTERY_IDX, :], features)

---
# Best Practices für die Visualisierung

> *„Bevor wir modellieren, müssen wir die Daten verstehen.“*

Wir werden den Herzkrankheitsdatensatz mit [Matplotlib](https://matplotlib.org/) untersuchen und ihn dann vorverarbeiten, damit er für maschinelles Lernen bereit ist.

Rufen wir zuerst die Funktionen aus, die uns eine Zusammenfassung aller Merkmale geben.

In [ ]:
print_feature_info(x, features)

print_missing_values(x, features)

<img alt="Balkendiagramm" height="184" width="200" src="https://openclipart.org/download/303250/1529053658.svg" style="float: right;margin-left:10px" />

## Balkendiagramm

Balkendiagramme (auch als Säulendiagramme bezeichnet) sind visuelle Hilfsmittel, die kategoriale Daten mithilfe rechteckiger Balken darstellen, wobei die Länge oder Höhe jedes Balkens proportional zu dem Wert ist, den er repräsentiert. Sie werden primär dazu verwendet, Mengen über verschiedene Gruppen oder Kategorien hinweg zu vergleichen, wodurch Muster und Trends leichter zu erkennen sind als in rohen Datentabellen.

Balkendiagramme werden in Matplotlib mithilfe der Methode [`bar`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.bar.html) für vertikale Balken oder [`barh`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.barh.html) für horizontale Balken erstellt.

Balkendiagramme lassen sich auf folgende Weise anpassen und erweitern:
- **Gestapelte Balken**: Wir können einen Satz von Werten auf einen anderen stapeln mithilfte des Parameters `bottom` in einem zweiten Aufruf von `bar`.
- **Gruppierte Balkendiagramme**: Durch Berechnung der Offset-Positionen und der Balkenbreite (`bar_width`) können wir die Balken verschiedener Gruppen innerhalb derselben Kategorie nebeneinander platzieren.
- **Fehlerbalken**: Unsicherheit wird mithilfe des Parameters `yerr` hinzugefügt.
- **Wertebeschriftungen**: Wir können die genauen Werte z.B. oberhalb jedes Balkens mihilfe der Methode [`text`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.text.html) anzeigen.
- **Visuelle Anpassungen**: Die Methode `bar()` gibt eine Liste von [`BarContainer`](https://matplotlib.org/stable/api/container_api.html#matplotlib.container.BarContainer)-Objekten zurück, was eine weitere Anpassung einzelner Balken über Methoden wie `set_color()` oder `set_alpha()` ermöglicht.

Extrahieren wir zunächst die Indizes und Namen aller kategorialen Merkmale.

In [ ]:
# Kategoriale Merkmale identifizieren
categorical_features = []  # [(Index, Name)]
for feature_index, (feature_name, info) in enumerate(features.items()):
    if len(info) != 1:
        categorical_features.append((feature_index, feature_name))
del feature_index, feature_name, info
print(f'Anzahl der kategorialen Merkmale: {len(categorical_features)}')

In [ ]:
print(categorical_features)

Die folgende Funktion erstellt ein Balkendiagramm auf der Grundlage der Messwerte eines einzelnen Merkmals. Testen wir sie für einpaar Merkmale.

In [ ]:
def plot_categorical_bar(ax, values: np.ndarray, categories: list[str]):
    """
    Stellt die Häufigkeiten jeder Kategorie eines Merkmals grafisch dar.
    
    :param ax: Axes-Objekt, mit dem die Grafik erstellt wird.
    :param values: Beobachtete Werte als eindimensionaler Integer-Array.
    :param categories: Liste der Kategorienamen.
    """
    positions = np.arange(len(categories))
    heights = np.bincount(values, minlength=len(categories))
    ax.bar(positions, heights, width=0.9)  # align='center'
    ax.set(axisbelow=True, ylabel='Anzahl', xticks=positions)
    ax.set_xticklabels(categories, fontsize=9, ha='right', rotation=45)
    ax.grid(axis='y', color='#A0A0A0', linestyle='--', linewidth=0.5)


def get_categorical(values: np.ndarray, y: np.ndarray) ->\
    tuple[np.ndarray, np.ndarray]:
    """
    Extrahiert die Werte der angegebenen kategorialen Merkmale als Indizes.

    :param values: originale Werte im Merkmalmatrix für das Merkmal als
                   Array der Form `(n,)`.
    :param y: Ausgabewerte als Integer-Array der Form `(n,)`.
    :return: Tupel der Form `(values, outcomes)` wobei:
             - `values` ein Integer-Array mit Kategorieindizes ist, und
             - `outcomes` ein Integer-Array mit Ausgabewerten ist.
    """
    missing = np.isnan(values)
    if np.any(missing):
        values = values[~missing]
        y = y[~missing]
    return values.astype(np.uint16), y


index_categorical = 4
info = categorical_features[index_categorical]
values, _ = get_categorical(x[:, info[0]], y)
categories = features[info[1]]
figure, ax = plt.subplots(figsize=(1 + len(categories), 3), dpi=100)
plot_categorical_bar(ax, values, categories)
ax.set_title(info[1])
figure.tight_layout()
plt.show(figure)
del index_categorical, info, values, _, categories, figure, ax

In [ ]:
th_scan = x[:, 12]
print(th_scan.dtype)
print(th_scan.shape)
print(f'Minimaler Wert: {np.nanmin(th_scan)}')
print(f'Maximaler Wert: {np.nanmax(th_scan)}')
print()

th_scan = x[:, 12].astype(np.uint16)
print(th_scan.dtype)
print(th_scan.shape)
print(f'Minimaler Wert: {np.nanmin(th_scan)}')
print(f'Maximaler Wert: {np.nanmax(th_scan)}')
print(np.isnan(th_scan).sum())
print()

### Übung: Gruppierte Balkendiagramm

Erweitern wir die obige Funktion, um ein gruppiertes Balkendiagramm zu erstellen, das die Beobachtungen für jede Kategorie in *gesund* und *krank* unterteilt.

In [ ]:
def plot_categorical_bar(ax, values: np.ndarray, categories: list[str],
                         y: np.ndarray):
    """
    Stellt die Häufigkeiten jeder Kategorie eines Merkmals grafisch dar.

    :param ax: Axes-Objekt, mit dem die Grafik erstellt wird.
    :param values: Beobachtete Werte als eindimensionaler Integer-Array
                   der Form `(n,)`.
    :param categories: Liste der Kategorienamen.
    :param y: Ausgabevektor der Form `(n,)`, der die Werte für "gesund"
              (`0`) und "krank" (`1`) speichert.
    """
    y_labels = ['gesund', 'krank']
    y_colors = ['#80DEEA', '#EF9A9A']
    widths = 0.4
    positions = np.arange(len(categories))
    xs = [positions - widths, positions]
    for index_outcome, outcome in enumerate(y_labels):
        heights = np.bincount(values[y == index_outcome], minlength=len(categories))
        ax.bar(xs[index_outcome], heights, align='edge', facecolor=y_colors[index_outcome],
               label=outcome, width=widths) 
    ax.legend()
    ax.set(axisbelow=True, ylabel='Anzahl', xticks=positions)
    ax.set_xticklabels(categories, fontsize=9, ha='right', rotation=45)
    ax.grid(axis='y', color='#A0A0A0', linestyle='--', linewidth=0.5)


index_categorical = 7
info = categorical_features[index_categorical]
values, y_current = get_categorical(x[:, info[0]], y)
categories = features[info[1]]
figure, ax = plt.subplots(figsize=(1 + len(categories), 3), dpi=100)
plot_categorical_bar(ax, values, categories, y_current)
ax.set_title(info[1])
figure.tight_layout()
plt.show(figure)
del index_categorical, info, values, y_current, categories, figure, ax

### Übung: Balkendiagramme für alle Merkmale

Verwenden Sie die oben definierte Funktion `plot_categorical_bar`, um Balkendiagramme für alle Merkmale in einem kombinierten Plot.

In [ ]:
row_count = 2
column_count = len(categorical_features) // row_count
figure, axes = plt.subplots(row_count, column_count, figsize=(16, 8), dpi=100)

for ax, info in zip(axes.flatten(), categorical_features):
    values, y_current = get_categorical(x[:, info[0]], y)
    categories = features[info[1]]
    plot_categorical_bar(ax, values, categories, y_current)
    ax.set_title(info[1])

figure.tight_layout()
plt.show(figure)
del row_count, column_count, figure, axes, ax, info, values, y_current, categories

<img alt="boxplot" height="180" width="180" src="https://openclipart.org/download/303249" style="float:right;margin-left:10px;margin-top:20px" />

## Kreisdiagramm

Ein Kreisdiagramm ist eine kreisförmige statistische Grafik, die in Abschnitte unterteilt ist, um numerische Proportionen darzustellen, wobei die Fläche jedes Abschnitts die Menge einer bestimmten Kategorie im Verhältnis zum Ganzen darstellt.

In Matplotlib, Kreisdiagramme werden von einem `Axes`-Objekt mittels der Methode [`pie`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.pie.html) erstellt. Als primäres Argument erwartet sie die Datenwerte. Wie bei anderen Visualisierungen, gibt es optionale Parameter für Beschriftungen, Farben, Explosion (um Slices zu versetzen) und mehr.

Wenn das Seitenverhältnis auf "gleich" eingestellt ist (`aspect='equal'`), wird sichergestellt, dass der Kreis als perfekter Kreis gezeichnet wird.

In [ ]:
def plot_categorical_pie(axes, values: np.ndarray, categories: list[str],
                         y: np.ndarray):
    """
    Stellt die Häufigkeiten jeder Kategorie eines Merkmals grafisch dar.

    :param axes: Liste von 2 Axes-Objekten, mit denen die Grafik erstellt
                 wird.
    :param values: Beobachtete Werte als eindimensionaler Integer-Array
                   der Form `(n,)`.
    :param categories: Liste der Kategorienamen.
    :param y: Ausgabevektor der Form `(n,)`, der die Werte für "gesund"
              (`0`) und "krank" (`1`) speichert.
    """
    y_labels = ['gesund', 'erkrankt']
    for i, label in enumerate(y_labels):
        frequencies = np.bincount(values[y == i], minlength=len(categories))
        axes[i].pie(frequencies, labels=categories)
        axes[i].set(aspect='equal', title=label)


index_categorical = 7
info = categorical_features[index_categorical]
values, y_current = get_categorical(x[:, info[0]], y)
categories = features[info[1]]
figure, axes = plt.subplots(1, 2, figsize=(7, 3.2), dpi=100)
plot_categorical_pie(axes, values, categories, y_current)
figure.suptitle(info[1])
figure.tight_layout()
plt.show(figure)

Kreisdiagramme werden in der Datenwissenschaft aufgrund ihrer Einschränkungen selten verwendet:

- *Schlechte Genauigkeit*: Menschen beurteilen Längen (Balkendiagramme) viel genauer als Winkel oder Flächen; Untersuchungen zeigen, dass Kreisdiagramme zu den am häufigsten missverstandenen Visualisierungen gehören, weil es für Betrachter schwierig ist, Segmente nach Wert zu ordnen oder Segmente ähnlicher Größe zu unterscheiden.
- *Unordnung bei vielen Kategorien*: Sie werden unleserlich und irreführend, wenn mehr als fünf oder sechs Kategorien angezeigt werden, da die Slices zu klein werden, um effektiv zu unterscheiden oder zu vergleichen.
- *Unfähigkeit, Trends anzuzeigen*: Kreisdiagrammen fehlt eine Zeitachse, wodurch sie für die Anzeige von Veränderungen im Zeitverlauf oder komplexen hierarchischen Daten ungeeignet sind.
- *Verzerrungsrisiken*: Die Verwendung von 3D-Effekten, explodierten Schnitten oder unterschiedlichen Ausrichtungen kann die Schnittgrößen optisch verzerren und zu einer Fehlinterpretation der Daten führen.

---
<img alt="Histogramm" height="371" width="508" src="https://upload.wikimedia.org/wikipedia/commons/7/7c/Torverteilung.png" style="float: right;margin-left:10px" />

## Histogramm

Ein **Histogramm** zeigt, wie Werte eines einzelnen numerischen Merkmals verteilt sind. Die Methode zum Zeichnen eines Histogramms ist [hist](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.hist.html).

Wir schauen uns zunächst an, wie viel (und welche) die numerischen Merkmale sind.

Der darauffolgende Codeausschnitt erstellt eine einzelne Abbildung mit 5 Histogrammen – eines pro numerischem Merkmal.

In [ ]:
# Numerische Merkmale identifizieren
numeric_features = []  # [(Index, Name)]
for feature_index, (feature_name, info) in enumerate(features.items()):
    if len(info) == 1:
        numeric_features.append((feature_index, feature_name))

figure, axes = plt.subplots(1, len(numeric_features), figsize=(15, 2.8), dpi=100)
for ax, (feature_index, feature_name) in zip(axes, numeric_features):
    ax.hist(x[:, feature_index], bins=20, color='steelblue', edgecolor='white')
    ax.set(axisbelow=True, title=feature_name, xlabel=feature_name, ylabel='Anzahl')
    ax.grid(axis='y', color='#A0A0A0', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show(figure)

### Übung: Histogramme nach Klasse

Erstellen Sie eine einzelne Figur mit 5 Nebenhandlungen (eine pro numerischem Merkmal).
Überlagern Sie in jedem Unterdiagramm **zwei Histogramme** – eines für Klasse `0` (gesund, blau) und eines für Klasse `1` (Herzkrankheit, rot) – und verwenden Sie `alpha=0.6` für Transparenz.

Fügen Sie zu jedem Nebenplot eine Legende hinzu, die zeigt, welche Farbe zu welcher Klasse gehört.

In [ ]:
# Beobachtungsindizes für jede Ausgabekategorie extrahieren
group_indices = [  # (Kategorie, Farbe, Indizes)
    ('gesund', 'steelblue', np.where(y == 0)[0]),
    ('krank', 'indianred', np.where(y == 1)[0])]

# Histogramme zeichnen
figure, axes = plt.subplots(1, len(numeric_features), figsize=(15, 2.8), dpi=100)
for ax, (feature_index, feature_name) in zip(axes, numeric_features):
    for state, color, sample_indices in group_indices:
        values = x[sample_indices, feature_index]
        ax.hist(values, bins=20, alpha=0.5, color=color, edgecolor='white', label=state)
    ax.set(axisbelow=True, title=feature_name, xlabel=feature_name, ylabel='Anzahl')
    ax.grid(axis='y', color='#A0A0A0', linestyle='--', linewidth=0.5)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show(figure)

<img alt="boxplot" height="280" width="200" src="https://openclipart.org/download/244707/Boxplot.svg" style="float: right;margin-left:10px" />

## Boxplots: Ausreißer auf einen Blick erkennen

Der Boxplot fasst die wichtigsten Merkmale der Daten zusammen. Die Linie innerhalb des Kästchens markiert den Median, also den Mittelwert des Datensatzes. Die Box selbst enthält die mittleren 50 % aller Beobachtungen, vom unteren Quartil Q1 (25. Perzentil) bis zum oberen Quartil Q3 (75. Perzentil). Die "Schnurrhaare" reichen bis zu Werten außerhalb dieses Bereichs und zeigen, wie weit sich die Daten ausbreiten. Einzelne Punkte jenseits der Whiskers werden häufig als Ausreißer betrachtet, d. h. als ungewöhnlich hohe oder niedrige Werte im Vergleich zu den übrigen Daten.

Den Code-Snippet unten erstellt ein Diagramm pro numerischen Merkmal, dass die Verteilungen für gesunden und kranken Patienten mittels Boxplots visualisiert.

In [ ]:
plot_size = (2 * len(numeric_features) + 1, 3.5)
figure, axes =\
    plt.subplots(1, len(numeric_features), figsize=plot_size, dpi=100)
for ax, (feature_index, feature_name) in zip(axes, numeric_features):
    states = []
    colors = []
    values = []
    for state, color, indices in group_indices:
        states.append(state)
        colors.append(color)
        v = x[indices, feature_index]
        values.append(v[~np.isnan(v)])
    bp = ax.boxplot(values, tick_labels=states, patch_artist=True, widths=0.7)
    for box_patch, median_patch, color in zip(bp['boxes'], bp['medians'], colors):
        box_patch.set(alpha=0.6, facecolor=color)
        median_patch.set(color='black')
    ax.set(axisbelow=True, title=feature_name)
    ax.grid(axis='y', color='#A0A0A0', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

## Hausaufgabe: Geigenplot

Der Geigenplot bietet eine Visualisierung ähnlich wie die eines Histogramms. Versuchen Sie die Verteilungen aller numerischen Merkmale für gesunde und kranke Patienten mittels [`violin`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.violin.html) zu vergleichen.

## Streudiagramme

Streudiagramme zeigen einzelne Messungen als Punkte. Die $X$-Koordinate jedes Punkts ist sein Wert für ein Merkmal und die $Y$-Koordinate - der Wert für ein anderes Merkmal. Die Methode zum Erstellen von Streudiagrammen ist [`scatter`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.scatter.html).

Ein Streudiagramm lässt uns erkennen, ob zwei Merkmale ein gemeinsames Muster aufweisen. Wenn wir Kategorien mit Punktfarben kodieren, können wir prüfen, ob sich das Muster zwischen den Klassen unterscheidet.

In [ ]:
def plot_scatter(x: np.ndarray, group_indices: list,
                 feature_x: tuple[int, str], feature_y: tuple[int, str]) -> plt.Figure:
    """
    Erstellt ein Streudiagramm unter Verwendung zweier Merkmale des Datensatzes.
    
    :param x: Merkmalmatrix als Array der Form `(Beobachtungen, Merkmale)`.
    :param group_indices: Gruppierung der Beobachtungen in Form einer Liste von Gruppendefinitionen.
    :param feature_x: das für die x-Achse zu verwendende Merkmal, als Tupel der Form `(Index, Name)`.
    :param feature_y: das für die y-Achse zu verwendende Merkmal, als Tupel der Form `(Index, Name)`.
    """
    figure, ax = plt.subplots(figsize=(5, 4), dpi=100)
    for state, color, indices in group_indices:
        xs, ys = x[indices, feature_x[0]], x[indices, feature_y[0]]
        ax.scatter(xs, ys, c=color, label=state, alpha=0.75, edgecolors='white', s=30)
    ax.set(axisbelow=True, xlabel=feature_x[1], ylabel=feature_y[1])
    ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)
    ax.legend()
    return figure


feature_x, feature_y = numeric_features[3], numeric_features[0]
figure = plot_scatter(x, group_indices, feature_x, feature_y)
plt.tight_layout()
plt.show()

### Übung: Streudiagrammmatrix (Paardiagramm)

Erstellen Sie ein **5×5-Raster von Nebenhandlungen** für die numerischen Merkmale.
- Außerdiagonale Zellen: **Streudiagramme** von Merkmalspaaren, gefärbt nach Ziel.
- Diagonale Zellen: **Histogramme** des Features, gefärbt nach Ziel (überlagert).

Dies ist im Wesentlichen ein *Paardiagramm*, der mit Matplotlib von Grund auf erstellt wird.

In [ ]:
n = len(numeric_features)

figure, axes = plt.subplots(n, n, figsize=(12, 12), dpi=100)
colors = {0: 'steelblue', 1: 'indianred'}

for i, feature_y in enumerate(numeric_features):
    for j, feature_x in enumerate(numeric_features):
        ax = axes[i, j]
        if i == j:
            # Diagonal: Histogramm
            for outcome_index, c in colors.items():
                ax.hist(x[y == outcome_index, feature_x[0]],
                        bins=20, alpha=0.5, color=c, edgecolor='none')
        else:
            # Außerhalb der Diagonalen: Streudiagramm
            for outcome_index, c in colors.items():
                sample_indices = y == outcome_index
                ax.scatter(x[sample_indices, feature_x[0]],
                           x[sample_indices, feature_y[0]],
                           c=c, alpha=0.5, s=15, edgecolors='none')

        ax.set(axisbelow=True)
        ax.grid(color='#A0A0A0', linestyle='--', linewidth=0.5)

        # Achsenbeschriftungen nur in der "Spalte" bzw. letzten "Reihe"
        if j == 0:
            ax.set_ylabel(feature_y[1])
        if i == n - 1:
            ax.set_xlabel(feature_x[1])

plt.tight_layout()
plt.show(figure)

## 1.5 Korrelations-Heatmap

Die *Pearson-Korrelationsmatrix* ist eine symmetrische quadratische Matrix, die die [Produkt-Moment-Korrelation](https://de.wikipedia.org/wiki/Korrelationskoeffizient_nach_Bravais-Pearson) zwischen jedem Variablenpaar in der Merkmalsmatrix $X$ anzeigt. Für einen Datensatz mit $d$ Variablen (Merkmalen) hat die Matrix die Größe $d \times d$, wobei jede Zelle $(i, j)$ den Koeffizienten $\rho_{ij}$ enthält, der die lineare Beziehung zwischen der Variablen $i$ und der Variablen $j$ quantifiziert. Denken Sie daran:
- Korrelation $\ne$ Kausalität,
- Die Korrelationsmatrix erfasst nur *lineare* Assoziationen.

Wir können die Numpy-Funktion [`corrcoef`](https://numpy.org/doc/stable/reference/generated/numpy.corrcoef.html) verwenden, um die Korrelationsmatrix zu berechnen. <span style="color:red">Wichtig: Diese Funktion berechnet Korrelationen zwischen den **Zeilen** ihres Parameters, daher müssen wir die *transponierte* Merkmalsmatrix übergeben.</span>

Eine Möglichkeit, eine Korrelationsmatrix zu visualisieren, ist ein Bild von $d \times d$ Pixeln, in dem jedes Pixel ein farbcodierter Wert der jeweiligen Zelle in der Matrix ist.

Die Matplotlib-Funktion zum Anzeigen eines Bildes ist [`imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html). Die Bibliothek bietet eine große Auswahl an [Zuordnungen von Werten zu Farben](https://matplotlib.org/stable/gallery/color/colormap_reference.html).

In [ ]:
# "Merkmalmatrix" aus allen numerischen Merkmalen + die Ausgabe konstruieren
feature_indices = np.array([i for i, _ in numeric_features], dtype=np.uint32)
xy = np.column_stack((x[:, feature_indices], y))
variable_names = [n for _, n in numeric_features] + ['Stand']

# Korrelationskoeffiziente berechnen
correlations = np.corrcoef(xy.T)

figure, ax = plt.subplots(figsize=(7, 6), dpi=100)
d = correlations.shape[0]
im = ax.imshow(correlations, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set(title='Korrelations-Heatmap', xticks=range(d), yticks=range(d))
ax.set_xticklabels(variable_names, rotation=45, ha='right')
ax.set_yticklabels(variable_names)

# Pixel (Zellen) mit Korrelationswerten annotieren
for i in range(d):
    for j in range(d):
        v = correlations[i, j]
        text_color = 'white' if abs(v) > 0.5 else 'black'
        ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                color=text_color, fontsize=9)

figure.colorbar(im, ax=ax, shrink=0.8, label='Pearson-Korrelation')
plt.tight_layout()
plt.show()

**Überlegen Sie:**
- Welches Merkmal hat die stärkste Korrelation mit der Zielvariable?
- Gibt es Merkmale, die stark miteinander korrelieren? Wenn ja, könnte das Probleme verursachen?

---
# Datenvorverarbeitung

## Umgang mit fehlenden Werten

Wir haben schon festgestellt, dass 2 Spalten in der Merkmalmatrix fehlende Werte beinhalten. Sehen wir, wo genau diese Lücken liegen.

In [ ]:
print_missing_values(x, features)

# Fehlende Werte pro Zeile berichten
print()
missing_per_row = np.sum(np.isnan(x), axis=1)
for missing_count, row_count in enumerate(np.bincount(missing_per_row)):
    print(f'Beobachtungen mit {missing_count} fehlenden Wert(en): {row_count}')

# Zielvariablewerten für die Beobachtungen mit fehlenden Werten berichten
i_missing = np.where(missing_per_row != 0)[0]
print(f'Ausgabewerte bei Beobachtungen mit fehlenden Werten: {y[i_missing]}')

### Übung: Fehlende Werte ergänzen

Es fehlen Werte in den Spalten für `Gefärbte Hauptgefäße` und `Thallium-Herzscan`. Eine Möglichkeit, mit fehlenden Werten umzugehen, ist die [Imputation](https://de.wikipedia.org/wiki/Imputation_(Statistik)) - füllen Sie diese leeren Zellen basierend auf anderen Werten, die für die jeweiligen Merkmale und/oder anderen Eigenschaften der jeweiligen Beobachtungen beobachtet wurden.

Erstellen Sie eine bereinigte Kopie der Merkmalmatrix mit dem Namen `x_clean`, wobei:
1. Fehlende Werte in `Gefärbte Hauptgefäße` werden mit dem [**Median**](https://de.wikipedia.org/wiki/Median) der Spalte aufgefüllt.
2. Fehlende Werte in `Thallium-Herzscan` werden mit dem [**Modus**](https://de.wikipedia.org/wiki/Modus_(Statistik)) (dem häufigsten Wert) aufgefüllt.

Drucken Sie anschließend zur Bestätigung die Anzahl der fehlenden Werte aus.

#### Tipps

1. Der *Median* eines Arrays kann mithilfe von [`median`](https://numpy.org/doc/stable/reference/generated/numpy.median.html) oder [`nanmedian`](https://numpy.org/doc/stable/reference/generated/numpy.nanmedian.html) ermittelt werden. Werfen Sie einen Blick in die Dokumentation, um zu sehen, worin der Unterschied zwischen diesen beiden Funktionen besteht.
2. Der *Modus* eines Integer-Arrays mit nicht-negativen Werten kann durch die Kombination der Funktionen [`bincount`](https://numpy.org/doc/stable/reference/generated/numpy.bincount.html) und [`argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) ermittelt werden.

In [ ]:
x_clean = x.copy()

# Fehlende Werte in 'Gefärbte Hauptgefäße' mit Median auffüllen
feature_names = list(features.keys())
index_feature = feature_names.index('Gefärbte Hauptgefäße')
fill_value = np.nanmedian(x_clean[:, index_feature])
x_clean[np.isnan(x_clean[:, index_feature]), index_feature] = fill_value

# Fehlende Werte in 'Thallium-Herzscan' mit Modus auffüllen
index_feature = feature_names.index('Thallium-Herzscan')
values = x_clean[:, index_feature]
values = values[~np.isnan(values)].astype(np.uint16)
fill_value = np.argmax(np.bincount(values))
x_clean[np.isnan(x_clean[:, index_feature]), index_feature] = fill_value

# Fehlende Werte nach der Imputation berichten
print_missing_values(x_clean, features)

## Feature-Skalierung: Standardisierung und Normalisierung

Viele Algorithmen des maschinellen Lernens reagieren empfindlich auf die Skalierung von Merkmalen. Zwei gängige Ansätze sind:

| Methode | Formel | Effekt |
|---------|--------|--------|
| **Standardisierung** (z-score) | $(x - \mu) / \sigma$ | $\mu = 0, \sigma = 1$ |
| **Normalisierung** (min-max) | $(x - x_{\min}) / (x_{\max} - x_{\min})$ | $\in [0, 1]$ |

Schauen wir uns den Effekt visuell an.

In [ ]:
def plot_transformations(values: np.ndarray, feature_name: str) -> plt.Figure:
    """
    Konstruiert Histogramme der ursprünglichen Merkmalswerte und ihrer Transformationen.

    :param values: Original-Merkmalswerte.
    :param feature_name: Name oder Beschreibung des Merkmals.
    :return: das neu erstellte Diagramm als `Figure`-Objekt.
    """

    scaler = StandardScaler()
    v_standard = scaler.fit_transform(values)
    scaler = MinMaxScaler()
    v_minmax = scaler.fit_transform(values)

    figure, axes = plt.subplots(1, 3, figsize=(9, 3.2), dpi=100, sharey='all')
    axes[0].hist(values, bins=20, color='steelblue', edgecolor='white')
    axes[0].set(axisbelow=True, title='Ursprunglich', xlabel=feature_name, ylabel='Anzahl')
    axes[1].hist(v_standard, bins=20, color='seagreen', edgecolor='white')
    axes[1].set(axisbelow=True, title='Standardisiert (z-Wert)')
    axes[2].hist(v_minmax, bins=20, color='darkorange', edgecolor='white')
    axes[2].set(axisbelow=True, title='Normalisiert (Min-Max)')
    for ax in axes:
        ax.grid(axis='y', color='#A0A0A0', linestyle='--', linewidth=0.5)
    return figure


feature_name = 'Serumcholesterin'
feature_description = f'{feature_name} ({features[feature_name][0]})'
index_feature = list(features.keys()).index(feature_name)
values = x[:, index_feature].reshape(-1, 1)
figure = plot_transformations(values, feature_description)
plt.tight_layout()
plt.show(figure)

# Begriffe

| *Deutsch*      | *Englisch*   |
|:--------------:|:------------:|
| Balkendiagramm | bar chart    | 
| Kreisdiagramm  | pie chart    |
| Histogramm     | histogram    |
| Boxplot        | boxplot      |
| Geigenplot     | violin plot  |
| Streudiagramm  | scatter plot |
| Paardiagramm   | pair plot    |
| Median         | median       |
| Modus          | mode         |